# Predict Seagrass at different times
* Designed to work year by year
* Pick a location and run

In [17]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio
import datetime

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:49682/status,
Dashboard: http://127.0.0.1:49682/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:49683,Workers: 0
Dashboard: http://127.0.0.1:49682/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:49700,Total threads: 2
Dashboard: http://127.0.0.1:49701/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:49686,


# Value to edit

In [3]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
large_seagrass_sites =  ["Ihutai", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "LeftBank_Nov25", "ThePoint_Nov25"]
large_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua",
                "LeftBank_Nov25", "LeftBank_Feb26", "ThePoint_Nov25", "ThePoint_Feb26"]


In [5]:
satellite_classes_all = {'Seagrass': 1, 'Seagrass submerged': 2, 'Gracilaria': 3, 'Gracilaria submerged': 4, 
                           'Ulva': 5, 'Submerged vegetation': 9, 'Microphytobenthos': 10, 'Satmarsh': 14,
                           'Unvegetated': 15, 'Water': 16, 'Terrestrial': 18, 'Rock': 19, 'Mixed': 22,}
satellite_classes_all_unet = {'Seagrass': 0, 'Seagrass submerged': 1, 'Gracilaria': 2, 'Gracilaria submerged': 3, 
                           'Ulva': 4, 'Submerged vegetation': 5, 'Microphytobenthos': 6, 'Satmarsh': 7,
                           'Unvegetated': 8, 'Water': 9, 'Terrestrial': 10, 'Rock': 11, 'Mixed': 12,}

In [12]:
sample_method = "sampling_2"
method_2_threshold = .8
model_max_cloud_cover = 1 # percentage
model_low_tide_delta_hrs = 0
model_low_tide_delta_mins = 30
test_threshold = 0.1

predict_max_cloud_cover = 0 # percentage"
predict_low_tide_delta_hrs = 0
predict_low_tide_delta_mins = 30


model_description = f"RF_model_ihutai_trained_13_class_{int(test_threshold*100)}_percent_test" # f"RF_model_all_trained_13_class_{int(test_threshold*100)}_percent_test"

website_tab = (
    f"{model_description}_train_{utils.get_samples_folder(sample_method, method_2_threshold)}_"
    f"{utils.get_low_tide_max_cloud_name(model_low_tide_delta_hrs, model_low_tide_delta_mins, model_max_cloud_cover)}_"
    f"predict_{utils.get_low_tide_max_cloud_name(predict_low_tide_delta_hrs, predict_low_tide_delta_mins, predict_max_cloud_cover)}"
)

In [13]:
model_path = utils.get_models_path(sample_method=sample_method, method_2_threshold=method_2_threshold,
                                   low_tide_delta_hrs=model_low_tide_delta_hrs, low_tide_delta_mins=model_low_tide_delta_mins,
                                   max_cloud_cover=model_max_cloud_cover)
model_file = model_path / model_description / "model.joblib"
prediction_sites = ["Ihutai"] #large_sites # all_training_sites
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [14]:
data_path = utils.get_data_path()
utils.create_data_folders()
predictions_path, satellite_path = utils.get_prediction_path(
    sample_method, method_2_threshold,
    model_max_cloud_cover=model_max_cloud_cover,
    predict_max_cloud_cover=predict_max_cloud_cover,
    predict_low_tide_delta_hrs=predict_low_tide_delta_hrs,
    predict_low_tide_delta_mins=predict_low_tide_delta_mins,
    model_low_tide_delta_hrs=model_low_tide_delta_hrs,
    model_low_tide_delta_mins=model_low_tide_delta_mins
)

predictions_path = predictions_path / model_description
predictions_path.mkdir(exist_ok=True, parents=True)

In [8]:
for prediction_site in prediction_sites:
    for year in years:
        print(f"{prediction_site}: year {year}")
    
        satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
        site_polygon_file = utils.get_site_polygon_path(prediction_site)
    
        if not satellite_file.exists():
            print(f"\tSave satellite image of the site {prediction_site} around lowtide without cloud in the year {year}")
            site_polygon = geopandas.read_file(site_polygon_file)
            satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                geometry=site_polygon,
                max_cloud_cover=predict_max_cloud_cover,
                year=year,
                low_tide_delta_hrs=predict_low_tide_delta_hrs,
                low_tide_delta_mins=predict_low_tide_delta_mins,
            )
            if len(satellite_data['time']) == 0:
                print("Warning: No satellite data without cloud. Ignore")
            else:
                print("\t\tSave")
                utils.write_netcdf_conventions_in_place(satellite_data)
                satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                utils.write_netcdf_conventions_in_place(satellite_data)
                utils.save_netcdf(satellite_data, satellite_file)
            utils.rgb_from_satellite(satellite_data_path=satellite_file)

CatlinsLake: year 2016
CatlinsLake: year 2017
	Save satellite image of the site CatlinsLake around lowtide without cloud in the year 2017
	Time from low tide: 0:29:58.974000 for S2A_MSIL2A_20170626T223701_R072_T59GLJ_20210210T015542
	Time from low tide: 0:07:28.974000 for S2A_MSIL2A_20171120T222531_R029_T59GLJ_20201014T210953
	Time from low tide: 0:20:00.973000 for S2B_MSIL2A_20170810T223659_R072_T59GLJ_20210210T223634
	Time from low tide: 0:13:41.026000 for S2A_MSIL2A_20171021T222541_R029_T59GLJ_20201015T011921
	Time from low tide: 0:26:28.974000 for S2A_MSIL2A_20171220T222531_R029_T59GLJ_20201019T193033
	Time from low tide: 0:12:30.973000 for S2B_MSIL2A_20171205T222529_R029_T59GLJ_20201014T121240
	Time from low tide: 0:12:30.973000 for S2B_MSIL2A_20171205T222529_R029_T59GLJ_20201014T121240
	Low tide tiles: 6 from a total lowish cloud cover tiles of 104
	No cloud tiles: 0 from the low tide tiles of 6
Satellite data path C:\Local\repos\seagrass-detection\data\predictions\low_tide_delta

## Predict values

In [18]:
for prediction_site in prediction_sites:

    ## Predict values
    for year in years:
        print(f"{prediction_site}: year {year}")
        prediction_file = predictions_path / f"{prediction_site}_{year}.nc"
        polygon_file = utils.get_site_polygon_path(prediction_site)
        satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
        
        if not prediction_file.exists():
            model_feature_names_file = model_file.with_name("feature_names.csv")
            if not satellite_file.exists():
                print("\tWARNING no satellite file to predict. Rerun the above cell if this is unexpected."
                      " Otherwise this indicates no suitable low-tide and low-cloud dates in the year.")
                continue
            predictions, satellite_data = training.predict_site(
                test_satellite_file=satellite_file,
                polygon_file=polygon_file,
                model_file=model_file,
                model_feature_names_file=model_feature_names_file)
            utils.save_netcdf(predictions, prediction_file)
        else:
            predictions = utils.load_classification(filename=prediction_file, chunks=None)
    
        for target_name in ["Seagrass", "Ulva", "Gracilaria"]:
            target_file = predictions_path /  f"{prediction_site}_{year}_{target_name.lower()}.gpkg"
            if not target_file.exists():
                target = {"date": [], "geometry": []}
                prediction = utils.load_classification(filename=prediction_file)
                prediction = utils.ensure_grid_indexing(prediction)
                target_id = pandas.read_csv(model_file.with_name("class_mappings.csv"), index_col=0)
                target_id = int(target_id.loc[target_id.index == target_name,"satellite_class_id"].squeeze())
                for time_index in range(len(prediction["time"])):
                    mask = predictions.isel(time=time_index) == target_id
                    target["date"].append(str(mask.time.data))
                    geometry = utils.mask_to_polygons(mask).dissolve().geometry
                    target["geometry"].append(geometry[0])
                target = geopandas.GeoDataFrame(target, crs="2193")
                target.to_file(target_file)

    ## Format for website
    output_folder = data_path / "website" / website_tab / prediction_site
    output_folder.mkdir(exist_ok=True, parents=True)

    if (output_folder / "seagrass_presence_absence_map.gpkg").exists():
        print(f"{prediction_site} already run. Go to next")
        continue
    extents_all_years = {"Seagrass": [], "Ulva": [], "Gracilaria": []}
    info = {"ids": [], "B04_scale":[], "B03_scale":[], "B02_scale": [], "date":[]}
    for year in years:
        print(f"{prediction_site}: year {year}")
        satellite_info_set = False
        for target in extents_all_years.keys():
            target_file = predictions_path / f"{prediction_site}_{year}_{target}.gpkg"

            if not target_file.exists():
                print(f"WARNING no {target_file.name} file. Run 'predict_for_site.ipynb' is this is unexpected.")
                continue
            target_extents = geopandas.read_file(target_file)
            extents_all_years[target].append(target_extents)

            for index, row in target_extents.iterrows():
                target_extent = geopandas.GeoDataFrame(geometry=[row["geometry"]], crs=utils.CRS_NZTM)
                date_YYMMDD = datetime.datetime.strptime(row.date, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%d')
        
                filename = f"{date_YYMMDD}_{target.lower()}.gpkg"
                target_extent.to_file(output_folder / filename)

                # Look up the satellite tile information
                if not satellite_info_set:
                    #print(f"\tGet Tile(s) ID: {date_YYMMDD}")
                    site_polygon = geopandas.read_file(utils.get_site_polygon_path(prediction_site))
                    tile_id, precentage_dict = sentinel2.get_satellite_info(geometry=site_polygon, date_YYMMDD=date_YYMMDD)
                    
                    info["date"].append(row.date)
                    info["ids"].append(tile_id)
                    info["B04_scale"].append(precentage_dict["B04"])
                    info["B03_scale"].append(precentage_dict["B03"])
                    info["B02_scale"].append(precentage_dict["B02"])
            satellite_info_set = True
            
    info = pandas.DataFrame(info)
    info = info.rename(columns={"ids": "Satellite Tile IDs",
                                "B04_scale": "B04 Rescale",
                                "B03_scale": "B03 Rescale",
                                "B02_scale": "B02 rescale"})
    
    for key, value in extents_all_years.items():
        print(f"\tTarget {key}")
        target_extents = pandas.concat(value, ignore_index=True)
        target_extents['area'] = target_extents.area
        info[f"{key} Area [m^2]"] = target_extents.area
        target_extents.dissolve()[["geometry"]].to_file(output_folder / f"{key.lower()}_presence_absence_map.gpkg")

    
    info.to_csv(output_folder / "info_all_dates.csv", index=False)
                

Ihutai: year 2016
Ihutai: year 2017
	WARNING no satellite file to predict. Rerun the above cell if this is unexpected. Otherwise this indicates no suitable low-tide and low-cloud dates in the year.
Ihutai: year 2018
	WARNING no satellite file to predict. Rerun the above cell if this is unexpected. Otherwise this indicates no suitable low-tide and low-cloud dates in the year.
Ihutai: year 2019
	WARNING no satellite file to predict. Rerun the above cell if this is unexpected. Otherwise this indicates no suitable low-tide and low-cloud dates in the year.
Ihutai: year 2020
	WARNING no satellite file to predict. Rerun the above cell if this is unexpected. Otherwise this indicates no suitable low-tide and low-cloud dates in the year.
Ihutai: year 2021
	WARNING no satellite file to predict. Rerun the above cell if this is unexpected. Otherwise this indicates no suitable low-tide and low-cloud dates in the year.
Ihutai: year 2022
	WARNING no satellite file to predict. Rerun the above cell if t

ValueError: No satellite items with valid RGB pixels at this site on 2016-04-06

In [ ]:
sample_method = "sampling_2" # sampling_1 sampling_2
method_2_thresholds = [.8, .5] # .95 .97 .98 .99 1.0
model_max_cloud_covers = [0, 1, 5, 10] # percentage
model_tide_deltas = {1: {"hrs": 0, "mins": 30}, 
                     2: {"hrs": 0, "mins": 45}, 
                     3: {"hrs": 1, "mins": 0}}
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
test_threshold = 0.1

predict_max_cloud_cover = 1 # percentage"
predict_tide_delta_hrs = 0
predict_tide_delta_mins = 30

model_name = f"RF_model_all_trained_13_class_{int(test_threshold*100)}_percent_test.joblib"

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()

In [ ]:
for model_max_cloud_cover in model_max_cloud_covers:
    for model_tide_delta in model_tide_deltas.values():
        for method_2_threshold in method_2_thresholds:

            ## Setup
            website_tab = (
                f"{model_description}_train_{utils.get_samples_folder(sample_method, method_2_threshold)}_"
                f"{utils.get_low_tide_max_cloud_name(model_tide_delta['hrs'], model_tide_delta['mins'], model_max_cloud_cover)}_"
                f"predict_{utils.get_low_tide_max_cloud_name(predict_tide_delta_hrs, predict_tide_delta_mins, predict_max_cloud_cover)}"
            )

            model_path = utils.get_models_path(
                sample_method=sample_method,
                method_2_threshold=method_2_threshold,
                low_tide_delta_hrs=model_tide_delta['hrs'],
                low_tide_delta_mins=model_tide_delta['mins'],
                max_cloud_cover=model_max_cloud_cover
            )
            model_file = model_path / model_description / "model.joblib"

            predictions_path, satellite_path = utils.get_prediction_path(
                sample_method, method_2_threshold,
                model_max_cloud_cover=model_max_cloud_cover,
                predict_max_cloud_cover=predict_max_cloud_cover,
                predict_low_tide_delta_hrs=predict_low_tide_delta_hrs,
                predict_low_tide_delta_mins=predict_low_tide_delta_mins,
                model_low_tide_delta_hrs=model_tide_delta['hrs'],
                model_low_tide_delta_mins=model_tide_delta['mins']
            )
            
            predictions_path = predictions_path / model_description
            predictions_path.mkdir(exist_ok=True, parents=True)


            for prediction_site in prediction_sites:

                ## Predict values
                for year in years:
                    print(f"{prediction_site}: year {year}")
                    prediction_file = predictions_path / f"{prediction_site}_{year}.nc"
                    polygon_file = utils.get_site_polygon_path(prediction_site)
                    satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
                    
                    if not prediction_file.exists():
                        model_feature_names_file = model_file.with_name("feature_names.csv")
                        if not satellite_file.exists():
                            print("\tWARNING no satellite file to predict. Rerun the above cell if this is unexpected."
                                  " Otherwise this indicates no suitable low-tide and low-cloud dates in the year.")
                            continue
                        predictions, satellite_data = training.predict_site(
                            test_satellite_file=satellite_file,
                            polygon_file=polygon_file,
                            model_file=model_file,
                            model_feature_names_file=model_feature_names_file)
                        utils.save_netcdf(predictions, prediction_file)
                    else:
                        predictions = utils.load_classification(filename=prediction_file, chunks=None)
                
                    for target_name in ["Seagrass", "Ulva", "Gracilaria"]:
                        target_file = predictions_path /  f"{prediction_site}_{year}_{target_name.lower()}.gpkg"
                        if not target_file.exists():
                            target = {"date": [], "geometry": []}
                            prediction = utils.load_classification(filename=prediction_file)
                            prediction = utils.ensure_grid_indexing(prediction)
                            target_id = pandas.read_csv(model_file.with_name("class_mappings.csv"), index_col=0)
                            target_id = int(target_id.loc[target_id.index == target_name,"satellite_class_id"].squeeze())
                            for time_index in range(len(prediction["time"])):
                                mask = predictions.isel(time=time_index) == target_id
                                target["date"].append(str(mask.time.data))
                                geometry = utils.mask_to_polygons(mask).dissolve().geometry
                                target["geometry"].append(geometry[0])
                            target = geopandas.GeoDataFrame(target, crs="2193")
                            target.to_file(target_file)
            
                ## Format for website
                output_folder = data_path / "website" / website_tab / prediction_site
                output_folder.mkdir(exist_ok=True, parents=True)
            
                if (output_folder / "seagrass_presence_absence_map.gpkg").exists():
                    print(f"{prediction_site} already run. Go to next")
                    continue
                extents_all_years = {"Seagrass": [], "Ulva": [], "Gracilaria": []}
                info = {"ids": [], "B04_scale":[], "B03_scale":[], "B02_scale": [], "date":[]}
                for year in years:
                    print(f"{prediction_site}: year {year}")
                    satellite_info_set = False
                    for target in extents_all_years.keys():
                        target_file = predictions_path / f"{prediction_site}_{year}_{target}.gpkg"
            
                        if not target_file.exists():
                            print(f"WARNING no {target_file.name} file. Run 'predict_for_site.ipynb' is this is unexpected.")
                            continue
                        target_extents = geopandas.read_file(target_file)
                        extents_all_years[target].append(target_extents)
            
                        for index, row in target_extents.iterrows():
                            target_extent = geopandas.GeoDataFrame(geometry=[row["geometry"]], crs=utils.CRS_NZTM)
                            date_YYMMDD = datetime.datetime.strptime(row.date, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%d')
                    
                            filename = f"{date_YYMMDD}_{target.lower()}.gpkg"
                            target_extent.to_file(output_folder / filename)
            
                            # Look up the satellite tile information
                            if not satellite_info_set:
                                #print(f"\tGet Tile(s) ID: {date_YYMMDD}")
                                site_polygon = geopandas.read_file(utils.get_site_polygon_path(prediction_site))
                                tile_id, precentage_dict = sentinel2.get_satellite_info(geometry=site_polygon, date_YYMMDD=date_YYMMDD)
                                
                                info["date"].append(row.date)
                                info["ids"].append(tile_id)
                                info["B04_scale"].append(precentage_dict["B04"])
                                info["B03_scale"].append(precentage_dict["B03"])
                                info["B02_scale"].append(precentage_dict["B02"])
                        satellite_info_set = True
                        
                info = pandas.DataFrame(info)
                info = info.rename(columns={"ids": "Satellite Tile IDs",
                                            "B04_scale": "B04 Rescale",
                                            "B03_scale": "B03 Rescale",
                                            "B02_scale": "B02 rescale"})
                
                for key, value in extents_all_years.items():
                    print(f"\tTarget {key}")
                    target_extents = pandas.concat(value, ignore_index=True)
                    target_extents['area'] = target_extents.area
                    info[f"{key} Area [m^2]"] = target_extents.area
                    target_extents.dissolve()[["geometry"]].to_file(output_folder / f"{key.lower()}_presence_absence_map.gpkg")
            
                
                info.to_csv(output_folder / "info_all_dates.csv", index=False)